##### KoELECTRA

In [112]:
import torch

x = torch.rand(5000, 5000).cuda()
y = torch.mm(x, x)

print(y.device)

cuda:0


In [113]:
import gc
import os
import sys
import json
import re

# =========================
# CUDA / 메모리 초기화
# =========================
import torch

gc.collect()
torch.cuda.empty_cache()

# =========================
# 기본 라이브러리
# =========================
import pandas as pd
import torch.nn.functional as F

# =========================
# HuggingFace 계열 먼저 로드
# =========================
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

# =========================
# sklearn
# =========================
from sklearn.metrics import (
    accuracy_score,
    f1_score
)

# =========================
# sentence-transformers는 맨 마지막
# =========================
from sentence_transformers import (
    SentenceTransformer,
    util
)

In [114]:
# =========================================================
# GPU 확인 및 작업 경로 체크
# =========================================================

print("=" * 80)
print("환경 검증 및 GPU 확인")
print("=" * 80)

print("현재 파이썬 작업 경로(PWD):", os.getcwd())
print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))
print("=" * 80)

환경 검증 및 GPU 확인
현재 파이썬 작업 경로(PWD): c:\NLP
CUDA 사용 가능: True
GPU 이름: NVIDIA GeForce RTX 3070


In [115]:
# =========================================================
# device 설정
# SBERT는 CPU
# KoELECTRA는 GPU
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# =========================================================
# 메모리 정리
# =========================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# =========================================================
# 모델 설정
# =========================================================

MODEL_NAME = "monologg/koelectra-small-v3-discriminator"

# =========================================================
# SBERT 모델
# GPU 사용
# =========================================================

similarity_model = SentenceTransformer(
    "snunlp/KR-SBERT-V40K-klueNLI-augSTS",
    device=device
)

# =========================================================
# 중요 패턴
# =========================================================

important_patterns = [
    "중요", "핵심", "반드시", "시험", "출제", "필수", 
    "정의", "의미", "란", "특징", "역할", "구조", 
    "원리", "과정", "개념", "주의"
]

# =========================================================
# hard negative 패턴
# =========================================================

hard_negative_patterns = [
    "최근", "설명하였다", "발표하였다", "소개하였다", 
    "진행하였다", "언급하였다", "보고하였다"
]

# =========================================================
# 문장 분리
# =========================================================

def split_sentences(text):

    if text is None:
        return []

    text = str(text)

    text = text.replace("\r", "\n")

    sentences = re.split(
        r'(?<=[.!?다])\s+|\n+',
        text
    )

    return [
        s.strip()
        for s in sentences
        if len(s.strip()) > 2
    ]

# =========================================================
# 핵심 개념 추출
# =========================================================

def extract_keywords(sentence):
    keywords = []
    patterns = [
        r"[가-힣A-Za-z0-9]+ 모델",
        r"[가-힣A-Za-z0-9]+ 알고리즘",
        r"[가-힣A-Za-z0-9]+ 함수",
        r"[가-힣A-Za-z0-9]+ 구조",
        r"[가-힣A-Za-z0-9]+ 기법",
        r"[가-힣A-Za-z0-9]+ 시스템",
        r"[가-힣A-Za-z0-9]+ 네트워크",
        r"[가-힣A-Za-z0-9]+ 학습",
        r"[가-힣A-Za-z0-9]+ 분석",
        r"[A-Z]{2,}"
    ]

    for pattern in patterns:
        found = re.findall(pattern, sentence)
        for word in found:
            if word not in keywords:
                keywords.append(word)

    return keywords

# =========================================================
# 데이터셋 생성
# =========================================================

def build_dataset(jsonl_path, max_data=300):
    # 파일이 진짜 존재하는지 체크하는 안전장치
    if not os.path.exists(jsonl_path):
        print(f"❌ 에러: [{jsonl_path}] 파일을 찾을 수 없습니다. 경로를 다시 확인하세요.")
        sys.exit(1)

    dataset = []

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if idx >= max_data:
                break

            item = json.loads(line)
            input_text = item["input"]
            summary_text = item["output"]

            sentences = split_sentences(input_text)
            if len(sentences) == 0:
                continue

            # -------------------------------------------------
            # SBERT 임베딩
            # -------------------------------------------------
            sentence_embeddings = similarity_model.encode(
                sentences,
                batch_size=32,
                convert_to_tensor=True,
                show_progress_bar=False
            )

            summary_embedding = similarity_model.encode(
                summary_text,
                batch_size=32,
                convert_to_tensor=True,
                show_progress_bar=False
            )

            # -------------------------------------------------
            # 문장별 라벨 생성
            # -------------------------------------------------
            for s_idx, sentence in enumerate(sentences):
                similarity = util.cos_sim(
                    sentence_embeddings[s_idx],
                    summary_embedding
                ).item()

                rule_based = any(p in sentence for p in important_patterns)
                hard_negative = any(p in sentence for p in hard_negative_patterns)

                label = 1 if (
                    (similarity >= 0.35 or rule_based)
                    and not hard_negative
                ) else 0

                dataset.append({
                    "text": sentence,
                    "label": label
                })

            # -------------------------------------------------
            # 메모리 정리
            # -------------------------------------------------
            del sentence_embeddings
            del summary_embedding
            gc.collect()

    return pd.DataFrame(dataset)

# =========================================================
# 데이터셋 생성 시작 (★ 경로 수정: koelectra_project/ 추가)
# =========================================================

print("\ntrain 데이터 생성 중...")
train_df = build_dataset("koelectra_project/train.jsonl", max_data=300)

print("valid 데이터 생성 중...")
valid_df = build_dataset("koelectra_project/valid.jsonl", max_data=100)

print("test 데이터 생성 중...")
test_df = build_dataset("koelectra_project/test.jsonl", max_data=100)

print("\n데이터 생성 완료")
print(f"train 개수: {len(train_df)}")
print(f"valid 개수: {len(valid_df)}")
print(f"test 개수 : {len(test_df)}")

# =========================================================
# 클래스 균형 맞추기
# =========================================================

positive_df = train_df[train_df["label"] == 1]
negative_df = train_df[train_df["label"] == 0]

negative_df = negative_df.sample(
    n=min(len(negative_df), len(positive_df) * 2),
    random_state=42
)

train_df = pd.concat([positive_df, negative_df])
train_df = train_df.sample(frac=1, random_state=42)

print("\n라벨 분포")
print(train_df["label"].value_counts())

# =========================================================
# Dataset 변환
# =========================================================

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

# =========================================================
# 토크나이저
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================================
# 토큰화
# =========================================================

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# =========================================================
# torch format
# =========================================================

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
valid_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# =========================================================
# GPU 상태 확인
# =========================================================

print("\n")
print("=" * 80)
print("GPU 상태 확인")
print("=" * 80)

print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))
    print(
        "현재 GPU 메모리 사용량:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )


# =========================================================
# 모델 로드
# =========================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)
model.to(device)

# =========================================================
# 평가 함수
# =========================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)

    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1
    }

# =========================================================
# 학습 옵션
# =========================================================

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=20,

    fp16=True,

    dataloader_num_workers=4,
    dataloader_pin_memory=True,

    load_best_model_at_end=True
)

# =========================================================
# Trainer
# =========================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# =========================================================
# 학습 시작
# =========================================================

print("\n")
print("=" * 80)
print("KoELECTRA 학습 시작")
print("=" * 80)

trainer.train()

# =========================================================
# 최종 평가
# =========================================================

results = trainer.evaluate(test_dataset)

print("\n")
print("=" * 80)
print("최종 성능")
print("=" * 80)

print(f"Accuracy : {results['eval_accuracy']:.4f}")
print(f"F1 Score : {results['eval_f1']:.4f}")
print(f"Loss      : {results['eval_loss']:.4f}")

# =========================================================
# 모델 저장
# =========================================================

SAVE_PATH = "./koelectra_keysentence_model"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("\n모델 저장 완료")

# =========================================================
# 예측 함수
# =========================================================

def predict_sentence(sentence):
    model.eval()
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][1].item()

    return pred, confidence

# =========================================================
# 긴 문장 테스트
# =========================================================

print("\n")
print("=" * 80)
print("긴 문장 기반 모델 성능 테스트")
print("=" * 80)

sample_text = """
딥러닝이란 인간의 신경망 구조를 모방하여 데이터를 학습하는 머신러닝 기법으로,
대량의 데이터를 활용해 이미지 분류, 자연어 처리, 음성 인식 등의 다양한 문제를 해결할 수 있다.

CNN 모델은 이미지의 공간적 특징을 효과적으로 추출할 수 있기 때문에
컴퓨터 비전 분야에서 매우 중요한 역할을 수행하며,
시험에서도 자주 출제되는 핵심 개념 중 하나이다.

Transformer 구조는 Self-Attention 메커니즘을 기반으로 문장 내 단어 간의 관계를 계산하며,
기존 RNN 기반 모델보다 병렬 처리가 가능하다는 장점이 있다.

오늘은 날씨가 좋아서 친구들과 함께 카페에 가서 커피를 마셨고,
저녁에는 영화를 보며 시간을 보냈다.

GPU는 대규모 행렬 연산을 병렬로 처리할 수 있기 때문에
딥러닝 모델 학습 속도를 크게 향상시킬 수 있으며,
CUDA 환경 설정은 실습에서 매우 중요하게 다뤄진다.
"""

sentences = split_sentences(sample_text)

for idx, sentence in enumerate(sentences):
    pred, confidence = predict_sentence(sentence)

    print(f"\n[{idx+1}] 문장")
    print(sentence)
    print(f"\n핵심 문장 확률: {confidence:.4f}")

    if confidence >= 0.40:
        print("\n예측 결과:")
        print("핵심 문장")
        keywords = extract_keywords(sentence)
        print("\n핵심 개념:")
        print(keywords)
    else:
        print("\n예측 결과:")
        print("일반 문장")

    print("-" * 80)


train 데이터 생성 중...
valid 데이터 생성 중...
test 데이터 생성 중...

데이터 생성 완료
train 개수: 3898
valid 개수: 1248
test 개수 : 1237

라벨 분포
label
1    2794
0    1104
Name: count, dtype: int64


Map: 100%|██████████| 1237/1237 [00:00<00:00, 28388.89 examples/s]




GPU 상태 확인
CUDA 사용 가능: True
GPU 이름: NVIDIA GeForce RTX 3070
현재 GPU 메모리 사용량: 7.44 GB


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-small-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\NLP\venv\lib\site-packages\accelerate\accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
c:\NLP\venv\lib\site-packages\accelerate\accelerator.py:463: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is de



KoELECTRA 학습 시작


...

{'eval_loss': 0.4921739101409912, 'eval_accuracy': 0.7844551282051282, 'eval_f1': 0.8670291646070193, 'eval_runtime': 22.8377, 'eval_samples_per_second': 54.646, 'eval_steps_per_second': 6.831, 'epoch': 3.0}


100%|██████████| 1464/1464 [05:41<00:00,  4.29it/s]


{'train_runtime': 341.6404, 'train_samples_per_second': 34.229, 'train_steps_per_second': 4.285, 'train_loss': 0.506324163551539, 'epoch': 3.0}


100%|██████████| 155/155 [00:03<00:00, 41.14it/s]




최종 성능
Accuracy : 0.8205
F1 Score : 0.8883
Loss      : 0.4445

모델 저장 완료


긴 문장 기반 모델 성능 테스트

[1] 문장
딥러닝이란 인간의 신경망 구조를 모방하여 데이터를 학습하는 머신러닝 기법으로,

핵심 문장 확률: 0.8255

예측 결과:
핵심 문장

핵심 개념:
['신경망 구조', '머신러닝 기법', '데이터를 학습']
--------------------------------------------------------------------------------

[2] 문장
대량의 데이터를 활용해 이미지 분류, 자연어 처리, 음성 인식 등의 다양한 문제를 해결할 수 있다.

핵심 문장 확률: 0.8266

예측 결과:
핵심 문장

핵심 개념:
[]
--------------------------------------------------------------------------------

[3] 문장
CNN 모델은 이미지의 공간적 특징을 효과적으로 추출할 수 있기 때문에

핵심 문장 확률: 0.8077

예측 결과:
핵심 문장

핵심 개념:
['CNN 모델', 'CNN']
--------------------------------------------------------------------------------

[4] 문장
컴퓨터 비전 분야에서 매우 중요한 역할을 수행하며,

핵심 문장 확률: 0.7264

예측 결과:
핵심 문장

핵심 개념:
[]
--------------------------------------------------------------------------------

[5] 문장
시험에서도 자주 출제되는 핵심 개념 중 하나이다.

핵심 문장 확률: 0.6157

예측 결과:
핵심 문장

핵심 개념:
[]
--------------------------------------------------------------------------------

[6] 

In [ ]:
# =========================================================
# 학습 완료 후 성능 테스트 코드
# 긴 강의자료 스타일 문장 테스트
# =========================================================

print("\n")
print("=" * 100)
print("KoELECTRA 핵심 문장 판별 테스트")
print("=" * 100)

sample_text = """

딥러닝은 인간의 신경망 구조를 모방하여 데이터를 학습하는 머신러닝 기법으로,
대량의 데이터를 활용해 이미지 분류, 자연어 처리, 음성 인식 등의 다양한 문제를 해결할 수 있다.
특히 최근에는 생성형 인공지능 기술이 발전하면서 딥러닝의 중요성이 더욱 강조되고 있다.

CNN 모델은 이미지의 공간적 특징을 효과적으로 추출할 수 있기 때문에
컴퓨터 비전 분야에서 매우 중요한 역할을 수행한다.
합성곱 연산을 활용하여 특징을 추출하며,
Pooling Layer를 통해 차원을 축소하고 연산량을 감소시킨다.
시험에서는 CNN의 구조와 특징이 자주 출제되는 핵심 개념 중 하나이다.

Transformer 구조는 Self-Attention 메커니즘을 기반으로 문장 내 단어 간의 관계를 계산하며,
기존 RNN 기반 모델보다 병렬 처리가 가능하다는 장점이 있다.
특히 BERT와 GPT 같은 최신 자연어 처리 모델의 기반 구조로 사용되며,
최근 자연어 처리 분야에서 가장 중요한 핵심 개념으로 평가받고 있다.

데이터 전처리 과정에서는 결측치 제거, 정규화, 이상치 탐지 등의 작업이 수행된다.
이 과정은 모델의 성능을 향상시키기 위해 반드시 필요하며,
데이터 품질이 낮을 경우 모델의 정확도가 크게 감소할 수 있다.

GPU는 대규모 행렬 연산을 병렬로 처리할 수 있기 때문에
딥러닝 모델 학습 속도를 크게 향상시킬 수 있다.
특히 CUDA 환경 설정은 딥러닝 실습에서 매우 중요하게 다뤄지며,
GPU 메모리 관리 또한 모델 성능 최적화에 중요한 역할을 한다.

시험 문제에서는 Softmax 함수의 역할과 Cross Entropy Loss의 계산 과정을
비교하여 설명하는 문제가 자주 출제된다.
또한 활성화 함수의 종류와 특징을 비교하는 문제 역시 핵심 유형으로 자주 등장한다.

오늘은 날씨가 좋아서 친구들과 함께 카페에 가서 커피를 마셨고,
저녁에는 영화를 보며 시간을 보냈다.
주말에는 여행을 가기로 계획했으며,
맛집을 검색하면서 시간을 보내기도 했다.

최근 학교 축제에서는 다양한 공연과 이벤트가 진행되었으며,
학생들이 많이 참여하여 분위기가 매우 활발했다.
동아리 부스에서는 여러 가지 게임과 체험 활동이 운영되었다.

KoELECTRA는 ELECTRA 구조를 기반으로 학습된 한국어 언어 모델이며,
문장 분류와 감정 분석, 자연어 이해 분야에서 높은 성능을 보이는 대표적인 모델이다.
특히 적은 데이터셋에서도 우수한 성능을 보인다는 특징이 있다.

"""

# =========================================================
# 문장 분리
# =========================================================

sentences = split_sentences(sample_text)

# =========================================================
# 핵심 문장 판별
# =========================================================

for idx, sentence in enumerate(sentences):

    pred, confidence = predict_sentence(sentence)

    print("\n")
    print("=" * 100)
    print(f"[문장 {idx+1}]")
    print("=" * 100)

    print(sentence)

    print(f"\n핵심 문장 확률: {confidence:.4f}")

    # -----------------------------------------------------
    # 핵심 문장
    # -----------------------------------------------------

    if confidence >= 0.40:

        print("\n예측 결과:")
        print("핵심 문장")

        keywords = extract_keywords(sentence)

        print("\n핵심 개념:")

        if len(keywords) > 0:

            for keyword in keywords:

                print(f"- {keyword}")

        else:

            print("- 추출된 핵심 개념 없음")

    # -----------------------------------------------------
    # 일반 문장
    # -----------------------------------------------------

    else:

        print("\n예측 결과:")
        print("일반 문장")

    print("\n")

##### KoBART 생성형 요약 추가

In [ ]:
# =========================================================
# KoBART 생성형 요약 추가 코드
# KoELECTRA 코드는 수정하지 않음
# =========================================================

# 필요 라이브러리
# !pip install transformers sentencepiece -q

from transformers import (
    PreTrainedTokenizerFast,
    BartForConditionalGeneration
)

# =========================================================
# KoBART 모델 로드
# =========================================================

print("=" * 80)
print("KoBART 생성형 요약 모델 로드")
print("=" * 80)

kobart_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    "digit82/kobart-summarization"
)

kobart_model = BartForConditionalGeneration.from_pretrained(
    "digit82/kobart-summarization"
).to(device)

print("KoBART 로드 완료")
print("=" * 80)

# =========================================================
# 핵심 문장 추출 함수
# (KoELECTRA 기존 함수 그대로 사용)
# =========================================================

def extract_core_sentences(text, threshold=0.8):

    sentences = split_sentences(text)

    core_sentences = []

    for sentence in sentences:

        pred, confidence = predict_sentence(sentence)

        # 중요 문장만 선택
        if pred == 1 and confidence >= threshold:

            core_sentences.append(sentence)

    return core_sentences

# =========================================================
# KoBART 생성형 요약 함수
# =========================================================

def generate_summary(
    text,
    threshold=0.8,
    max_input_length=700
):

    # -----------------------------------------------------
    # 1. 핵심 문장 추출
    # -----------------------------------------------------

    core_sentences = extract_core_sentences(
        text,
        threshold=threshold
    )

    # -----------------------------------------------------
    # 2. 핵심 문장 없을 경우
    # -----------------------------------------------------

    if len(core_sentences) == 0:

        print("핵심 문장이 없어 원문 일부 사용")

        core_text = text[:max_input_length]

    else:

        # -------------------------------------------------
        # 문장 정제
        # -------------------------------------------------

        filtered = []

        for sent in core_sentences:

            sent = sent.strip()

            # 너무 짧은 문장 제거
            if len(sent) < 15:
                continue

            # 중복 제거
            if sent not in filtered:
                filtered.append(sent)

        # -------------------------------------------------
        # 입력 길이 제한
        # -------------------------------------------------

        core_text = ""

        for sent in filtered:

            if len(core_text) + len(sent) > max_input_length:
                break

            core_text += sent + " "

    # -----------------------------------------------------
    # 3. 토큰화
    # -----------------------------------------------------

    inputs = kobart_tokenizer(
        core_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    # -----------------------------------------------------
    # 4. 생성형 요약
    # -----------------------------------------------------

    summary_ids = kobart_model.generate(

        input_ids=input_ids,
        attention_mask=attention_mask,

        # 요약 길이
        max_length=180,
        min_length=60,

        # Beam Search
        num_beams=8,

        # 반복 방지
        repetition_penalty=1.8,
        no_repeat_ngram_size=3,

        # 자연스러운 길이
        length_penalty=1.2,

        early_stopping=True
    )

    # -----------------------------------------------------
    # 5. 디코딩
    # -----------------------------------------------------

    summary = kobart_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return core_sentences, summary

# =========================================================
# 테스트용 긴 문서
# =========================================================

long_text = """

딥러닝은 인간의 신경망 구조를 모방하여 데이터를 학습하는 머신러닝 기법으로,
대량의 데이터를 활용해 이미지 분류, 자연어 처리, 음성 인식 등의 다양한 문제를 해결할 수 있다.
특히 최근에는 생성형 인공지능 기술이 발전하면서 딥러닝의 중요성이 더욱 강조되고 있다.

CNN 모델은 이미지의 공간적 특징을 효과적으로 추출할 수 있기 때문에
컴퓨터 비전 분야에서 매우 중요한 역할을 수행한다.
합성곱 연산을 활용하여 특징을 추출하며,
Pooling Layer를 통해 차원을 축소하고 연산량을 감소시킨다.

Transformer 구조는 Self-Attention 메커니즘을 기반으로 문장 내 단어 간의 관계를 계산하며,
기존 RNN 기반 모델보다 병렬 처리가 가능하다는 장점이 있다.
특히 BERT와 GPT 같은 최신 자연어 처리 모델의 기반 구조로 사용된다.

데이터 전처리 과정에서는 결측치 제거, 정규화, 이상치 탐지 등의 작업이 수행된다.
이 과정은 모델의 성능을 향상시키기 위해 반드시 필요하다.

GPU는 대규모 행렬 연산을 병렬로 처리할 수 있기 때문에
딥러닝 모델 학습 속도를 크게 향상시킬 수 있다.
특히 CUDA 환경 설정은 딥러닝 실습에서 매우 중요하게 다뤄진다.

KoELECTRA는 ELECTRA 구조를 기반으로 학습된 한국어 언어 모델이며,
문장 분류와 감정 분석 분야에서 높은 성능을 보인다.

"""

# =========================================================
# 생성형 요약 실행
# =========================================================

core_sentences, summary = generate_summary(
    long_text,
    threshold=0.8
)

# =========================================================
# 결과 출력
# =========================================================

print("\n")
print("=" * 100)
print("추출된 핵심 문장")
print("=" * 100)

for idx, sentence in enumerate(core_sentences):

    print(f"\n[{idx+1}] {sentence}")

print("\n")
print("=" * 100)
print("최종 생성형 요약")
print("=" * 100)

print(summary)

print("\n")
print("=" * 100)
print("요약 완료")
print("=" * 100)

print("\n")
print("=" * 100)
print("최종 생성형 요약문")
print("=" * 100)

print(summary)



In [ ]:
import sys

print(sys.executable)

In [ ]:
import platform
import sys

print(sys.executable)
print(sys.version)

In [ ]:
import os

print(os.environ.get("PATH"))

##### KoELECTRA, KoBART, Qwen 모델 로드

In [116]:
import torch

device = "cuda"

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM
)

# =====================================================
# KoELECTRA
# =====================================================

electra_tokenizer = AutoTokenizer.from_pretrained(
    "./koelectra_keysentence_model"
)

electra_model = AutoModelForSequenceClassification.from_pretrained(
    "./koelectra_keysentence_model"
)

electra_model.to(device)
electra_model.eval()

print("KoELECTRA 로드 완료")

# =====================================================
# KoBART
# =====================================================
from transformers import (
    BartForConditionalGeneration,
    PreTrainedTokenizerFast
)

kobart_tokenizer = (
    PreTrainedTokenizerFast
    .from_pretrained(
        "gogamza/kobart-summarization"
    )
)

kobart_model = (
    BartForConditionalGeneration
    .from_pretrained(
        "gogamza/kobart-summarization"
    )
)

kobart_model.to(device)

print("KoBART 로드 완료")

# =====================================================
# Qwen
# =====================================================

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

qwen_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Qwen 로드 완료")


KoELECTRA 로드 완료


c:\NLP\venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BartTokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


KoBART 로드 완료


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
c:\NLP\venv\lib\site-packages\accelerate\utils\modeling.py:1341: UserWarning: Current model requires 603984384 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.62s/it]


Qwen 로드 완료


##### torch 확인

In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

In [ ]:
import torch

print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
print("KoELECTRA:", next(electra_model.parameters()).device)
print("KoBART:", next(kobart_model.parameters()).device)
print("Qwen:", next(qwen_model.parameters()).device)

##### PDF → PaddleOCR → KoELECTRA → KoBART → Qwen → 시험공부 노트

In [125]:
# =========================================================
# PDF → PaddleOCR → KoELECTRA → KoBART → Qwen → 시험공부 노트
# (Qwen 모델 로드 이후 코드)
# =========================================================

import re
import torch
from datetime import datetime

# =========================================================
# OCR 결과 파일 경로
# =========================================================

ocr_text_path = "ocr_output_260608_2052\ocr_result.txt"

with open(
    ocr_text_path,
    "r",
    encoding="utf-8"
) as f:

    all_text = f.read()

# =========================================================
# 한글 깨짐 보정
# =========================================================

replace_dict = {
    # =====================================================
    # 기본 OCR 오류
    # =====================================================

    "힘올": "힘을",
    "귀리": "쿼리",
    "귀리블": "쿼리를",
    "쿼리블": "쿼리를",
    "김색": "검색",

    "문서클": "문서를",
    "결과루": "결과를",
    "결과 틀": "결과를",

    "찾올": "찾을",
    "있올": "있을",

    # =====================================================
    # 프롬프트 관련
    # =====================================================

    "프롭프트": "프롬프트",
    "프루프트": "프롬프트",
    "프#프트": "프롬프트",
    "프 콤프트": "프롬프트",
    "프콤프트": "프롬프트",
    "프롬프트틀": "프롬프트를",

    "엔지니어랗": "엔지니어링",
    "지니어랗": "엔지니어링",

    # =====================================================
    # 애플리케이션
    # =====================================================

    "애플리게이선올": "애플리케이션을",
    "애플리키이선올": "애플리케이션을",
    "애플리키이선": "애플리케이션",
    "애플리베이선": "애플리케이션",
    "애플리켜이선": "애플리케이션",

    # =====================================================
    # 조사 오류
    # =====================================================

    "능력올": "능력을",
    "작업올": "작업을",
    "답변올": "답변을",
    "출력올": "출력을",
    "출려올": "출력을",
    "모델올": "모델을",

    "사용자루": "사용자를",

    "사용자경험올": "사용자 경험을",
    "경험올": "경험을",

    "학습능력올": "학습 능력을",
    "전체과정올": "전체 과정을",

    "질문올": "질문을",

    "정보름": "정보를",
    "이유름": "이유를",
    "주제름": "주제를",
    "도구름": "도구를",

    "그것들올": "그것들을",
    "시스템올": "시스템을",
    "기능올": "기능을",

    "컨텍스트틀": "컨텍스트를",

    # =====================================================
    # LLM / NLP
    # =====================================================

    "권텍스트": "컨텍스트",
    "건텍스트": "컨텍스트",

    "취리블": "처리를",

    "LLMO|": "LLM이",

    # =====================================================
    # 문장 오류
    # =====================================================

    "것이상울": "것 이상을",
    "이상울": "이상을",
    "영향울": "영향을",

    "이해하러면": "이해하려면",
    "분석하러면": "분석하려면",

    "한계름": "한계를",

    "활까요": "할까요",

    "도메인.": "도메인,",

    "유용한엔드-투-엔드": "유용한 엔드-투-엔드",

    "출럭": "출력",

    "구축햇습니다": "구축했습니다",
    "햇습니다": "했습니다",

    "있없습": "있었습니다",
    "있었습니다 니다": "있었습니다",
    "없습니다 니다": "있었습니다",

    # =====================================================
    # 추가 OCR 패턴
    # =====================================================

    "다물 것입니다": "다룰 것입니다",
    "담구할 것입니다": "탐구할 것입니다",

    "언을 수": "얻을 수",

    "열럿습니다": "열렸습니다",
    "보켓습니다": "보겠습니다",
    "보완습니다": "보았습니다",

    "다툼니다": "다룹니다",

    "정렬원": "정렬된",

    "아기택처": "아키텍처",

    "덜 것입니다": "될 것입니다",

    "최점단": "최첨단",
    "논": "는",
    "김종": "검증",
    "월 수": "될 수",
    "되/습": "되었습니다",

    "담구": "탐구",
    "다물": "다룰",

    "첫봇": "챗봇",
    "붓": "봇",

    "현력적인": "협력적인",
    "동직하지": "동작하지",
    "맞추화된": "맞춤화된",

    "않울": "않을",
    "시작되엎는지틀": "시작되었는지를",
    "살펴봄으로씨": "살펴봄으로써",
    "이해틀": "이해를",
    "되엎는지": "되었는지",

    "RAC": "RAG",
    "프륭프트": "프롬프트",
    "폐르소나": "페르소나",
    "푸수 학습": "소수 학습",

    "글로드-3": "클로드-3",
    "근유 연성": "유연성",
    "맛추화하여": "맞춤화하여",
    "신 회성": "신뢰성",
    "굳렌즈": "콘텐츠",
    "매뉴일": "매뉴얼",
    "첫겨": "예제",
    "보젠습니다": "보겠습니다",
    "다툼 것입니다": "다룰 것입니다",
    "올 가르칠": "을 가르칠",
    "올 위한": "을 위한",
}

def fix_text(text):

    for k, v in replace_dict.items():
        text = text.replace(k, v)


    # OCR 자주 발생하는 오타 보정
    text = text.replace("텍스 트", "텍스트")
    text = text.replace("활용하 눈", "활용하는")
    text = text.replace("맛추화된", "맞춤화된")
    text = text.replace("상호 연결본", "상호 연결된")
    text = text.replace("굉크드인", "광고적인")
    text = text.replace("동작되지", "동작되지")

    text = re.sub(r"[ \t]+", " ", text)

    # 연속 줄바꿈 제거
    text = re.sub(r"\n{2,}", "\n", text)

    # 슬라이드 번호 제거
    text = re.sub(r"\b\d{2}\s+","",text)

    text = re.sub(r'([가-힣])([A-Z])', r'\1 \2', text)

    text = re.sub(r'([a-zA-Z])([가-힣])', r'\1 \2', text)

    text = re.sub(r"CHAPTER\s*\d*","",text)

    text = re.sub(r"\b\d+\s*$","",text)

    # 조사 OCR 오류 보정

    text = re.sub(r"([가-힣])올\b",r"\1을",text)

    text = re.sub(r"([가-힣])름\b",r"\1를",text)

    text = re.sub(r"([가-힣])블\b",r"\1를",text)

    text = re.sub(r"([가-힣])클\b",r"\1를",text)
 

    return text.strip()

# 추가
def remove_garbage_chars(text):

    # 한자 제거
    text = re.sub(
        r'[\u4e00-\u9fff]',
        ' ',
        text
    )

    # 일본어 제거
    text = re.sub(
        r'[\u3040-\u30ff]',
        ' ',
        text
    )

    # OCR 특수문자 제거
    text = re.sub(
        r'[㉠-㉯①-⑳]',
        ' ',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    return text


def is_valid_sentence(sentence):

    sentence = sentence.strip()

    stop_titles = [
        "자연어처리의 발전 과정",
        "텍스트 전처리와 토큰화",
        "텍스트 임베딩 기법의 진화",
        "딥러닝 기반 모델과 최신 언어 모델",
        "딥러닝 기반 모델과 최신 언어 모델까지",
        "자연어처리의 주요 Task",
        "Index 자연어처리의발전과정"
    ]

    if sentence in stop_titles:
        return False

    if len(sentence) < 10:
        return False
    
    if len(sentence.split()) < 2:
        return False
    
    # 제목성 문장 제거
    if sentence.startswith("■"):
        return False
    
    if sentence.startswith("["):
        return False

    special_count = len(
        re.findall(r"[，。：■□◆◇]", sentence)
    )

    if special_count > 3:
        return False

    # 한글이 하나도 없으면 제외
    if (
        not re.search(r"[가-힣]", sentence)
        and not re.search(r"[A-Za-z]", sentence)
    ):
        return False
    
    if sentence.count("（") >= 2:
        return False

    if sentence.count(")") >= 3:
        return False

    if len(re.findall(r"[■▲◆]", sentence)) > 1:
        return False
    

    if "PART" in sentence:
        return False

    if "CHAPTER" in sentence:
        return False

    if "그림" in sentence:
        return False
    
    if re.match(r"^\d+\s", sentence):
        return False

    if "첫 번째 단계" in sentence and len(sentence) < 40:
        return False
    
    if "###" in sentence:
        return False

    if "주관적:" in sentence:
        return False

    if "생각:" in sentence:
        return False

    if "행동:" in sentence:
        return False

    if "관찰:" in sentence:
        return False

    if "GPT-J" in sentence:
        return False

    if "FLAN-T5" in sentence:
        return False

    if "EleutherAI" in sentence:
        return False
    
    if "현재 재고" in sentence:
        return False

    if "USD 가격" in sentence:
        return False

    if "어시스템트 답변" in sentence:
        return False

    if "get" in sentence and len(sentence) < 50:
        return False
    
    if "첫 GPT:" in sentence:
        return False

    if "0 X" in sentence:
        return False

    if len(re.findall(r"[A-Za-z]", sentence)) > 30:
        return False


    english_words = len(
        re.findall(r"[A-Za-z]{3,}", sentence)
    )

    if english_words >= 5:
        return False


    # 영어가 한글보다 지나치게 많으면 제외
    hangul = len(
        re.findall(r"[가-힣]", sentence)
    )

    english = len(
        re.findall(r"[A-Za-z]", sentence)
    )

    total = hangul + english

    if total > 0:

        hangul_ratio = hangul / total

        if hangul_ratio < 0.15:
            return False
        
    # OCR 깨진 문장 제거
    bad_words = [
        "Io",
        "룹古",
        "놈곰",
        "극아공",
        "무애워긋료",
        "Io놀信",

        "엔지니어랗",
        "Fngincering",
        "romp",
        "spitti>",
        "허3",
        "PROPMET"
    ]

    if any(
        word in sentence
        for word in bad_words
    ):
        return False
    
    
    # ==========================
    # 예시 문장 제거
    # ==========================

    bad_examples = [
        "예시",
        "문제:",
        "답:",
        "컨텍스트 점수",
        "알리 봉고",
        "PDF 문서",
        "가봉 대통령",
        "연필",
        "묶음",
        "범고래",
        "고정비용",
        "[예제",
        "예제 4-",
        "A) 개",
        "B) 개",
        "C) 개",
        "D) 개",
        "첫겨",
        "매뉴일",
    ]

    if any(
        word in sentence
        for word in bad_examples
    ):
        return False


    if sentence.endswith("?"):
        return False

    if ":" in sentence and len(sentence) > 80:
        return False
    
    # ==========================
    # 번역/대화 예시 제거
    # ==========================

    example_words = [
        "예를 들어",
        "경우입니다",
        "점원:",
        "5-turbo",
        "영어에서 튀르키예어로",
        "이쪽입니다",
        "GPT-4용 시스템 프롬프트",
        "DB 에서 찾은",
        "문장을 영어에서",
    ]

    if any(
        word in sentence
        for word in example_words
    ):
        return False
        
    # 한글이 너무 적으면 제거
    #if hangul < 5:
    #    return False

    # 단어 수가 너무 적으면 제거
    #if len(sentence.split()) < 3:
    #    return False

    # 잘린 문장 제거
    if len(sentence) < 20 and sentence.endswith(("의", "및", "시", "패")):
        return False

    
    if len(sentence.split()) < 4:
        return False
    
    bad_patterns = [
        "Visnmen",
        "They They",
        "PART",
        "CHAPTER",
        "그림 3-",
        "그림 4-",
        "romp",
        "Fngincering",
        "en -0",
        "~end"
    ]

    if sum(
        p in sentence
        for p in bad_patterns
    ) >= 1:
        return False

    return True

# =========================================================
# PPT 잡음 제거
# =========================================================

noise_words = [

    "History of Programming Language",
    "Thank you",
    "-김정순",
]

def remove_noise(text):

    for word in noise_words:
        text = text.replace(word, "")

    return text

def clean_text(text):

    text = re.sub(
        r"\b\d{2}\s*",
        "",
        text
    )

    text = text.replace(
        "•",
        "\n- "
    )

    text = re.sub(
        r"([가-힣])([A-Z][a-z])",
        r"\1 \2",
        text
    )

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    return text.strip()

def remove_ocr_noise(text):

    lines = []

    for line in text.split("\n"):

        line = line.strip()

        if not line:
            continue

        hangul = len(
            re.findall(r"[가-힣]", line)
        )

        english = len(
            re.findall(r"[A-Za-z]", line)
        )

        if hangul < 5:
            continue

        bad_chars = len(
            re.findall(
                r"[㉠㉡㉢㉣個仰迅愁諭]",
                line
            )
        )       

        if bad_chars >= 3:
            continue

        # 영어가 너무 많으면 제거
        if english > hangul:
            continue

        lines.append(line)

    return "\n".join(lines)


def restore_ocr_text(text):

    text = text[:2500]

    prompt = f"""
너는 한국어 OCR 교정기이다.

규칙

1. OCR 오타만 수정한다.
2. 문맥상 명백한 OCR 오류만 수정한다.
3. 내용을 추가하지 않는다.
4. 요약하지 않는다.
5. 설명하지 않는다.
6. 문단 구조는 유지한다.
7. 기술서적 문맥에 맞게 수정한다.

예시

귀리 -> 쿼리
귀리블 -> 쿼리를
김색 -> 검색
프#프트 -> 프롬프트
프롭프트 -> 프롬프트
프롬프트 엔지니어랗 -> 프롬프트 엔지니어링
애플리게이선올 -> 애플리케이션을
애플리키이선올 -> 애플리케이션을
힘올 -> 힘을
결과루 -> 결과를
출력올 -> 출력을

OCR 결과

{text}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        chat_text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    inputs.pop("token_type_ids", None)

    inputs = {
        k: v.to(qwen_model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False
        )

    generated_ids = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    result = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return result.strip()


# =========================================================
# KoELECTRA 중요도 점수
# =========================================================

def get_score(text):

    try:

        inputs = electra_tokenizer(
            text,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        with torch.no_grad():

            outputs = electra_model(
                **inputs
            )

            score = torch.softmax(
                outputs.logits,
                dim=1
            )[0][1].item()

        return score

    except Exception as e:

        print("\n====================")
        print("KoELECTRA 오류")
        print(e)
        print("====================\n")

        return 0
    
# =========================================================
# KoBART 핵심 내용 정리
# =========================================================

def generate_summary(core_text):

    inputs = kobart_tokenizer(
        core_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    with torch.no_grad():

        summary_ids = kobart_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=256,
            min_length=30,
            num_beams=4,
            early_stopping=True
        )

    summary = kobart_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary

# =========================================================
# 키워드 추출
# =========================================================

def extract_keywords(text):

    print("새 extract_keywords 실행")

    concepts = [
        "LLM",
        "GPT",
        "RAG",
        "NLP",
        "프롬프트 엔지니어링",
        "프롬프트",
        "언어 모델",
        "미세 조정",
        "전이학습",
        "소수 학습",
        "연쇄적 사고",
        "페르소나",
        "의미 기반 검색",
        "검색 증강 생성",
        "애플리케이션",
        "도메인"
    ]

    keywords = []

    for concept in concepts:

        if concept in text and concept not in keywords:
            keywords.append(concept)

    return keywords[:5]

# =========================================================
# Qwen 노트 생성
# =========================================================

def generate_note(text):

    prompt = f"""

너는 시험공부 노트 작성기이다.

입력된 핵심 문장을 기반으로
시험공부용 노트를 작성하라.

규칙

1. 입력 문장에서 주요 개념을 추출한다.
2. 핵심 내용을 요약한다.
3. 입력에 없는 새로운 사실은 추가하지 않는다.
4. 시험공부 노트 형식으로 작성한다.

출력 형식

[주요 개념]
- 개념 : 설명

[핵심 내용]
- 내용
- 내용


입력:


{text}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        chat_text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    # Qwen 오류 방지
    inputs.pop("token_type_ids", None)

    inputs = {
        k: v.to(qwen_model.device)
        for k, v in inputs.items()
    }

    print("입력 길이 :", len(text))

    with torch.no_grad():

        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    result = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    # =====================================================
    # 출력 형식 통일
    # =====================================================

    result = re.sub(
        r"\[\s*핵\s*키워드\s*\]",
        "[핵심 키워드]",
        result
    )

    result = re.sub(
        r"\[\s*핵키워드\s*\]",
        "[핵심 키워드]",
        result
    )

    result = re.sub(
        r"■\s*핵심\s*아이.*?:",
        "[핵심 키워드]",
        result
    )

    result = re.sub(
        r"[\[\■]?\s*시험포인트\s*[\]\:]?",
        "[시험 포인트]",
        result
    )

    result = re.sub(
        r"[\[\■]?\s*주요 내용\s*[\]\:]?",
        "[주요 내용]",
        result
    )

    result = re.sub(
        r"\[핵심 개념\].*?(?=\[|$)",
        "",
        result,
        flags=re.DOTALL
    )

    result = re.sub(
        r"\[핵 아이디어\].*?(?=\[|$)",
        "",
        result,
        flags=re.DOTALL
    )

    # =====================================================
    # 기존 후처리
    # =====================================================


    result = re.sub(
        r"\n{3,}",
        "\n\n",
        result
    )

    # 중국어 제거
    result = re.sub(
        r'[\u4e00-\u9fff]+',
        '',
        result
    )

    # 일본어 제거
    result = re.sub(
        r'[\u3040-\u30ff]+',
        '',
        result
    )

    # 깨진 줄 제거
    lines = []

    for line in result.split("\n"):

        line = line.strip()

        if not line:
            continue

        if "。" in line:
            continue

        # 빈 괄호 제거
        if line in ["[]", "()", "（）"]:
            continue

        lines.append(line)

    result = "\n".join(lines)

    return result.strip()


# =========================================================
# 결과 저장 문자열
# =========================================================

result = ""

result += "=" * 80 + "\n"
result += "자연어처리 시험공부 노트\n"
result += "=" * 80 + "\n\n"

# =========================================================
# 페이지 분리
# =========================================================

pages = re.split(
    r"===== PAGE \d+ =====",
    all_text
)

pages = [
    p.strip()
    for p in pages
    if p.strip()
]

# =========================================================
# 페이지별 처리
# =========================================================

for page_idx, text in enumerate(pages):
    note = ""

    print("=" * 80)

    print(f"{page_idx+1}페이지 원문")
    print("=" * 80)
    print(text[:3000])
    print("=" * 80)

    text = clean_text(text)

    text = fix_text(text)

    text = remove_noise(text)

    print("OCR 복원 생략")

    text = remove_garbage_chars(text)

    text = fix_text(text)

    print("=" * 80)
    print("전처리 결과")
    print("=" * 80)
    print(text[:3000])

    text = remove_ocr_noise(text)

    print("=" * 80)
    print("remove_ocr_noise 이후")
    print("=" * 80)
    print(text[:3000])

    if len(text) < 20:
        continue

    print(f"{page_idx+1}페이지 처리중...")

    # =====================================================
    # 문장 분리
    # =====================================================

    sentences = []

    for line in re.split(
        r"[.!?\n]+",
        text
    ):

        line = line.strip()

        if len(line) < 15:
            continue

        sentences.append(line)

    print("=" * 80)
    print("문장 분리 결과")
    print("=" * 80)

    for s in sentences[:10]:
        print(s)

    print("문장 수:", len(sentences))
    print("=" * 80)

    # =====================================================
    # KoELECTRA
    # =====================================================

    scored = []

    for sentence in sentences:

        if not is_valid_sentence(sentence):
            continue

        # OCR 때문에 비정상적으로 긴 문장 제거
        if len(sentence) > 300:
            continue

        # 너무 짧은 문장 제거
        if len(sentence) < 40:
            continue

        # 문장 끝이 이상하면 제거
        if not sentence.endswith(("다", "니다", ".", "습니다")):
            continue

        # OCR 잘림 문장 제거
        bad_fragments = [
            "뉘양스",
            "시권스트",
            "니다:",
            "이라도",
            "통해",
        ]

        if any(sentence.endswith(x) for x in bad_fragments):
            continue

        score = get_score(sentence)

        scored.append(
            (
                score,
                sentence
            )
        )

    print("scored 개수 :", len(scored))

    scored.sort(
        key=lambda x: x[0],
        reverse=True
    )

    top_k = 5

    important = [

        sentence
        for score, sentence
        
        in scored[:top_k]
    ]

    important = list(
        dict.fromkeys(important)
    )

    important = sorted(
        important,
        key=lambda x: text.find(x)
    )

    important = important[:3]

    if len(important) == 0:

        print(
            f"{page_idx+1}페이지 중요 문장 없음"
        )

        continue


    print("문장 개수:", len(sentences))
    print("중요 문장 개수:", len(important))

    

    print("\n===== 중요 문장 =====")

    for s in important:
        print(s)

    print("=====================\n")

    # =====================================================
    # Qwen 입력
    # =====================================================

    important = [
        remove_garbage_chars(
            clean_text(s)
        )
        for s in important
    ]

    important_text = "\n".join(important)

    important_text = re.sub(
        r'[^\n가-힣A-Za-z0-9\s]',
        ' ',
        important_text
    )

    important_text = re.sub(
        r'\s+',
        ' ',
        important_text
    )

    # =====================================================
    # KoBART 입력 확인
    # =====================================================

    #print("=" * 80)
    #print("KoBART 입력")
    #print("=" * 80)
    #print(important_text)
    #print("=" * 80)

    # =====================================================
    # KoBART 요약
    # =====================================================

    #if len(important_text.strip()) > 0:

    #    summary = generate_summary(
    #        important_text
    #    )

    #else:

    #    summary = ""
    
    #study_text = important_text


    #print("=" * 80)
    #print("KoBART 요약")
    #print("=" * 80)
    #print(summary)
    #print("=" * 80)


    
    # =====================================================
    # Qwen 생성
    # =====================================================

    import time

    start = time.time()

    print("Qwen 생략")

    important_text = "\n".join(important)

    import re

    keywords = extract_keywords(important_text)

    print("새 extract_keywords 실행")
    print("important_text")
    print(important_text)

    print("keywords")
    print(keywords)
    
    print("=" * 80)
    print("키워드  결과")
    print("=" * 80)
    print(keywords)
    print("=" * 80)

    note += "[키워드]\n"

    for kw in keywords:
        note += f"- {kw}\n"

    note += "[주요 내용]\n"

    for sent in important:
        note += f"- {sent}\n"

    print("Qwen 종료")
    print(
        "Qwen 시간 :",
        round(time.time() - start, 1)
    )

    # =====================================================
    # 중요 문장 그대로 사용
    # =====================================================

    print("=" * 80)
    print(f"{page_idx+1}페이지 중요 문장")
    print("=" * 80)

    for sent in important:
        print(sent)


    print("=" * 80)
    print("키워드")
    print("=" * 80)

    for kw in keywords:
        print(kw)

    print("=" * 80)

    print()

    # ==========================
    # 출력 형식 보정
    # ==========================

    note = note.replace(
        "[핵심 키워드]",
        "\n[핵심 키워드]"
    )

    note = note.replace(
        "[주요 내용]",
        "\n[주요 내용]"
    )

    note = note.replace(
        "[KERNEL_KEYWORD]",
        "[핵심 키워드]"
    )

    note = note.replace(
        "[KERNEL KEYWORD]",
        "[핵심 키워드]"
    )

    note = note.replace(
        "[KERNEL KEYWORDS]",
        "[핵심 키워드]"
    )

    # =====================================================
    # 저장
    # =====================================================

    result += f"■ {page_idx+1}페이지\n\n"

    result += note

    result += "\n\n"

    result += "-" * 80

    result += "\n\n"

# =========================================================
# 파일 저장
# =========================================================

timestamp = datetime.now().strftime(
    "%y%m%d_%H%M"
)

output_file = (
    f"study_note_{timestamp}.txt"
)

with open(
    output_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(result)

# =========================================================
# 출력
# =========================================================

print(result[:5000])

print(
    f"\n저장 완료 : {output_file}"
)

1페이지 원문
CHAPTER 프#프트 엔지니어랗의 첫 번째 단계 3.1 들어가는 글 2장에서는 LLM의 힘올 활용하여 자연어 귀리블 통해 관련 문서클 찾올 수 있는 빠르고 효율 적인 비대칭 의미 기반 검색 시스템올 구축햇습니다. 이 시스템은 LLM이 방대한 양의 텍스 트로 사전 훈련된 덕분에 귀리 속의 숨은 의미까지 이해하고 정확한 결과루 김색할 수 있없습 니다. 그러나 효과적인 LLM 기반 애플리게이선올 구축하는 것은 사전 훈련된 모델올 연결하고 결과 틀 검색하는 것 이상울 필요로 할 수 있습니다: 만약 사용자 경험올 향상시키기 위해 그것들올 분석하러면 어떻게 해야 활까요? 또한 LLM의 학습 능력올 활용하여 전체 과정올 완성하고 유 용한 엔드-투-엔드en -0 '~end LLM 기반 애플리키이선올 생성하고자 할 수도 있습니다 . 이때가 바로 프롭프트 엔지니어랗이 등장하는 지점입니다. 3.2 프롭프트 엔지니어령 프루프트 엔지니어랗"romp Fngincering은 효과적으로 작업올 전달하여 정확하고 유용한 출력올 반 환하도록 유도하는 LLM에 대한 입력 (프롭프트)올 만드는 것입니다 (그림 3-1) . 프롭프트 엔 지니어랗에는 언어의 뉘양스. 작업 중인 특정 도메인. 그리고 사용 중인 LLM의 능력과 한계름 CHAPTER 3 프롭프트 엔지니어랗의 첫 번째 단계 97
OCR 복원 생략
전처리 결과
프롬프트 엔지니어링의 첫 번째 단계 3.1 들어가는 글 2장에서는 LLM 의 힘을 활용하여 자연어 쿼리를 통해 관련 문서를 찾을 수 있는 빠르고 효율 적인 비대칭 의미 기반 검색 시스템을 구축했습니다. 이 시스템은 LLM 이 방대한 양의 텍스트로 사전 훈련된 덕분에 쿼리 속의 숨은 의미까지 이해하고 정확한 결과를 검색할 수 있었습니다. 그러나 효과적인 LLM 기반 애플리케이션을 구축하는 것은 사전 훈련된 모델을 연결하고 결과를 검색하는 것 이상을 필요로 할 수 있습니다: 만약 사용자 경험을 향상시키기 위해 그것들을 분석하려면 어떻게 해야 할까요? 또한 LLM 의 학습 

In [109]:
print(text)

RAC 시스템과 AI 에이전트에 대한 탐구를 통해서 매략. 적응성 . 그리고 사용할 수 있는 도구 에 대한 깊은 이해의 중요성이라는 핵심 주제틀 다시 강조하고 싶습니다. 데이터베이스틀 활용 하여 답변의 근거지 마련하든. 사용자 쿼리를 해결하기 위해 디지털 도구를 활용하든 이러한 애플리게이선의 성공 여부는 LLM 의 생성 기능과 외부 데이터 소스 또는 도구의 특수성 및 신 회성 간의 미표한 균형에 달려 있습니다. 이제 우리는 AI 애플리케이션의 다음 단계를 내다보는 지점에 서 있습니다: 이 시점에 우리에 게 중요한 것은 이 여정이 현재 진행형임을 인식해야 합니다. AI 의 환경은 끊임없이 진화하 고 있으미. 기술과 사람의 요구가 교차하는 지점에 새로운 도전과 기회가 등장하고 있습니다. RAC 시스템과 AI 에이전트의 개발 및 평가에서 언은 통찰력은 종착점이 아니라 더 정고하고 공감하다 효과적인 AI 애플리케이션을 향한 디넘돌입니다. 앞으로의 장에서는 운리적 고려사항 기술적 장애물, AI 애플리게이선의 미지의 영역에 대해 더 자세히 살펴불 것입니다. 우리의 목표는 단순히 동작하는 AI 시스템을 만드는 것이 아니라 사람의 능력을 향상시키고 이해를 증진하다 궁극적으로 삶을 풍요롭게 하는 경험을 창출하는 것입니다. AI 생태계는 방대하고 다양하다 잠재력과 합정으로 가득 차 있습니다. 하지만 사러 깊은 접근 방식과 명확한 비전이 있다면 기술적으로 뛰어날 뿐만 아니라 의미 임고 영향력 F 는 슬루선을 구성하는 데 필요한 요소들을 구성할 수 있습니다. 이것이 바로 새로운 발견. 창의성. 지속적인 개선의 여정인 AI 애플리케이션의 핵심입니다. 6 PART LLM 소개
